# Analysis of the Strogatz problems

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(au.parse_if_needed)
full_report["sympy_defuzz"] = full_report.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
fr2 = full_report.set_index(["run_set", "data_set", "sample_num"])

In [ ]:
full_report.run_set.unique()

In [ ]:
srb1_key = "SRB-2026-06-25-1715-arr8"
srb2_key = "SRB-2026-07-13-1130"
cht1_key = "CHT-2026-07-02-1730"
cht2_key = "CHT-2026-07-13-1130"
srb1 = fr2.loc[srb1_key]
srb2 = fr2.loc[srb2_key]
cht1 = fr2.loc[cht1_key]
cht2 = fr2.loc[cht2_key]

In [ ]:
(fr2.groupby(level=["run_set", "data_set"]).size() == 32).all()

These are the best ones overall

In [ ]:
srb1_min_mse_ixs = srb1.groupby(level=["data_set"]).mse.idxmin()
srb2_min_mse_ixs = srb2.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()
cht2_min_mse_ixs = cht2.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2.loc[srb2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1.loc[cht1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht2.loc[cht2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

It looks like `srb1` is good, but `srb2` is better.
It looks like `cht1` is good but `cht2` is even better.
Especially for the `bacres` data sets.

In [ ]:
srb_key = srb2_key
srb = srb2
srb_min_mse_ixs = srb2_min_mse_ixs
cht_key = cht2_key
cht = cht2
cht_min_mse_ixs = cht2_min_mse_ixs

In [ ]:
main_data_sets = list(fr2.index.get_level_values("data_set").unique())
main_data_sets.remove("d_sfgrid1")
main_data_sets.remove("d_sfgrid2")
main_data_sets

In [ ]:
fr3 = fr2.loc[fr2.index.isin(main_data_sets, level="data_set")]

In [ ]:
srb_threshold_table = au.mse_threshold_table(srb)
srb_threshold_table

In [ ]:
srb_threshold_table.to_csv("Generated/srb_threshold_table.csv")
au.to_latex(
    srb_threshold_table,
    file="Generated/srb_threshold_table.tex",
    strip_colname_prefix="mse",
    strip_rowname_prefix="d_",
)

In [ ]:
cht_threshold_table = au.mse_threshold_table(cht)
cht_threshold_table

In [ ]:
cht_threshold_table.to_csv("Generated/cht_threshold_table.csv")
au.to_latex(
    cht_threshold_table,
    file="Generated/cht_threshold_table.tex",
    strip_colname_prefix="mse",
    strip_rowname_prefix="d_",
)

These are problems I ran extra samples of to get better reliability estimates.

In [ ]:
srb_extra_key = "SRB-2026-07-25-1900"
extra_problems = ["d_bacres1", "d_bacres2", "d_shearflow1"]
srb_extra = fr2.loc[([srb_key, srb_extra_key], extra_problems),:]

In [ ]:
srb_extra.sort_values("mse", ascending=True).head()

## Polynomials

In [ ]:
data_sets_polynomial = [
    "d_lv1",
    "d_lv2",
    "d_vdp1",
    "d_vdp2"
    ]

In [ ]:
srb.loc[data_sets_polynomial]

The best ones are exactly correct.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Only one sample is not a polynomial in $x$ and $y$.

In [ ]:
(srb.loc[data_sets_polynomial]
 .sympy_defuzz.apply(lambda e: not e.is_polynomial(sympy.abc.x, sympy.abc.y))
 .groupby(level="data_set")
 .sum())

All runs on all polynomial data sets are correct up to fuzz.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_polynomial])

In [ ]:
polynomial_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 29),
    "complexity_binwidth": 1,
    "mse_lims": (1.0e-32, 1.2e-20),
    "mse_binwidth": 0.5,
    "spiffy_titles": [r"Lotka-Volterra $x$", r"Lotka-Volterra $y$", r"Van der Pol $x$", r"Van der Pol $y$"],
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_polynomial],
    file_stem="srb-polynomial-complexity-mse-displot",
    **polynomial_plot_params
    )

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_polynomial],
    file_stem="cht-polynomial-complexity-mse-displot",
    **polynomial_plot_params
    )

## Rational functions

In [ ]:
data_sets_rational = [
    "d_bacres1",
    "d_bacres2",
    "d_predprey1",
    "d_predprey2"
    ]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

The majority of solutions are not rational functions.
Most of them are actually fine apart from fuzz and cruft.

In [ ]:
(srb.loc[data_sets_rational]
 .sympy_defuzz
 .apply(lambda e: not e.is_rational_function())
 .groupby(level="data_set").sum())

### Predator-prey

In [ ]:
srb.loc["d_predprey1", ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb.loc["d_predprey1"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

The predator-prey problem is well-behaved.
There are solution at MSE $8\times 10^{-9}$ and $1.9\times 10^{-8}$ with an imperfect denominator that's pretty close, so I'll use $2\times 10^{-8}$ as a cutoff.

In [ ]:
rational_threshold = 2.0e-8

In [ ]:
srb.loc["d_predprey2", ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb.loc["d_predprey2"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-3))

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational], threshold=rational_threshold)

In [ ]:
predprey1_ex_A = srb.loc[("d_predprey1", 17), "sympy"]
predprey1_ex_A

In [ ]:
predprey1_ex_A_pos = sympy.posify(au.replace_near_integer(predprey1_ex_A.evalf(), tolerance=1.0e-3))
predprey1_ex_A_pos[0]

In [ ]:
predprey1_ex_B = srb.loc[("d_predprey1", 12), "sympy"]
predprey1_ex_B

In [ ]:
au.replace_near_integer(predprey1_ex_B.evalf(), tolerance=2.0e-2)

### Bacterial respiration

These problems aren't so well behaved.

In [ ]:
srb.loc["d_bacres1"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-2))

In [ ]:
srb.loc["d_bacres2"].sympy.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-2))

Using the SRB configuration:
`bacres1` has a slight imperfection in a denominator, but is otherwise correct.

In [ ]:
srb_bacres1 = srb.loc[srb_min_mse_ixs].loc["d_bacres1", "sympy"].iloc[0]
srb_bacres1

In [ ]:
au.replace_near_integer(srb_bacres1, tolerance=1e-2)

In [ ]:
sympy.simplify(au.replace_near_integer(srb_bacres1, tolerance=1e-2))

`bacres2` is correct.

In [ ]:
srb_bacres2 = srb.loc[srb_min_mse_ixs].loc["d_bacres2", "sympy"].iloc[0]
srb_bacres2

In [ ]:
sympy.simplify(au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5))

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational], threshold=rational_threshold)

In [ ]:
bacres1_rf_ixs = srb.loc["d_bacres1"].sympy_defuzz.apply(lambda e: e.is_rational_function())

In [ ]:
srb.loc["d_bacres1"][bacres1_rf_ixs].sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

Using SRB config, these require more than the usual 32 samples to pin down.
Results with 192:

In [ ]:
srb_extra.loc[([srb_key, srb_extra_key], ["d_bacres1"]), :].sort_values("mse")

For `d_bacres1`, Looks like 3 perfect solutions and several decent approximations here.

In [ ]:
(srb_extra.loc[([srb_key, srb_extra_key], ["d_bacres1"]), :].sort_values("mse").iloc[0:10]
 .sympy_defuzz.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2)))

In [ ]:
srb_extra.loc[([srb_key, srb_extra_key], ["d_bacres2"]), :].sort_values("mse")

For `d_bacres2`, Looks like 6 perfect solutions and several decent approximations here.

In [ ]:
(srb_extra.loc[([srb_key, srb_extra_key], ["d_bacres2"]), :].sort_values("mse").iloc[0:10]
 .sympy_defuzz.apply(lambda e: sympy.simplify(au.replace_near_integer(e, tolerance=1.0e-2))))

### CHT configuration

Using the CHT configuration, all of these very best samples are correct.

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc["d_bacres1"].sort_values("mse")

In [ ]:
cht.loc["d_bacres2"].sort_values("mse")

In [ ]:
bacres1 = pd.read_csv("ode-strogatz/d_bacres1.txt")
bacres1

Curiously, this function has the wrong denominator everywhere:

In [ ]:
fsym = cht.loc["d_bacres1"].sort_values("mse").iloc[1].sympy
fsym

In [ ]:
sympy.expand(fsym)

For reference, the series for the actual function at $x = \infty$ is

$20 - x - \frac{2y}{x} + \frac{4y}{x^3} + O(x^{-5})$

And approximations to all of those terms appear in Jessamine's solution.

In [ ]:
f = sympy.lambdify((sympy.abc.x, sympy.abc.y), fsym)

The MSE is indeed very low, because there are no $x$ values near $0$ that would reveal that the denominator is wrong.

In [ ]:
points = pd.DataFrame({"x": bacres1.x, "y": bacres1.y, "z": bacres1.label, "z_hat": f(bacres1.x, bacres1.y)})
points["err"] = (points.z - points.z_hat)**2
points.sort_values("err", ascending=False).head(10)

In [ ]:
((f(bacres1.x, bacres1.y) - bacres1.label)**2).mean()

If we cheat, all of them have to be rational functions, and more are correct.

In [ ]:
au.count_by_threshold(cht.loc[data_sets_rational], threshold=rational_threshold)

Even with cheating, only one has the correct denominator.
Most of the rest are picking up some kind of series expansion.

In [ ]:
cht.loc["d_bacres1"].sort_values("mse").sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

In [ ]:
rational_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 399),
    "complexity_binwidth": 20,
    "mse_lims": (1.0e-32, 1.e-1),
    "mse_binwidth": 2.0,
    "spiffy_titles": [r"bac. resp. $x$", r"bac. resp. $y$", r"predator-prey $x$", r"predator-prey $y$"],
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_rational],
    file_stem="srb-rational-complexity-mse-displot",
    **rational_plot_params
    )

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_rational],
    file_stem="cht-rational-complexity-mse-displot",
    **rational_plot_params
    )

## Trigonometric functions, part 1

The shear flow problems are kind of a mess so we'll handle them separately

In [ ]:
data_sets_trig = [
    "d_barmag1",
    "d_barmag2",
    "d_glider1",
    "d_glider2",
    ]

These are all correct.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_trig, ["mse", "complexity_defuzz", "sympy_defuzz"]]

All samples are essentially correct.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_trig])

In [ ]:
trig_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0,29),
    "complexity_binwidth": 1,
    "mse_lims": (5.0e-29, 1.2e-26),
    "mse_binwidth": 0.1,
    "spiffy_titles": [r"bar magnet $x$", r"bar magnet $y$", r"glider $x$", r"glider $y$"],
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_trig],
    file_stem="srb-trig-complexity-mse-displot",
    **trig_plot_params
    )

## Trigonometric functions, part 2

The shear flow problems are kind of a mess so we'll handle them separately.
I decided not to go into analysis of the `sfgrid` data sets, which I generated by applying the vector field functions to an evenly spaced grid.
They are not part of the original data set.
In some other experiments, the `sfgrid` problem seemed to be easier to solve, but that turned out not to be the case in these benchmarks.

In [ ]:
data_sets_sf = [
    "d_shearflow1",
    "d_shearflow2",
    ]

For `shearflow2`, looks like about $17/32$ are correct.
For `shearflow1`, none are correct.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_sf],)

In [ ]:
shearflow_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 299),
    "complexity_binwidth": 10,
    "mse_lims": (1.0e-31, 1.e-1),
    "mse_binwidth": 1.0,
    "spiffy_titles": [r"shear flow $x$", r"shear flow $y$"],
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_sf],
    file_stem="srb-shearflow-complexity-mse-displot",
    **shearflow_plot_params
    )

If we cheat, there are a lot more correct answers.
The CHT configuration includes a full set of trig functions, including $\cot$, which saves Jessamine from having to build up $\cos / \sin$.

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_sf],
    file_stem="cht-shearflow-complexity-mse-displot",
    **shearflow_plot_params
    )

In [ ]:
au.count_by_threshold(cht.loc[data_sets_sf],)

Using the SRB configuration and running up to 192 trials, `shearflow1` still doesn't get solved.

In [ ]:
srb_extra.loc[([srb_key, srb_extra_key], ["d_shearflow1"]), :].sort_values("mse")

In [ ]:
shearflow1_best = srb_extra.loc[([srb_key, srb_extra_key], ["d_shearflow1"]), :].sort_values("mse").iloc[0]

In [ ]:
shearflow1_best

In [ ]:
au.replace_near_integer(shearflow1_best.sympy, tolerance=1.0e-2)